# Train the Hangman BiLSTM on Kaggle GPU

Clones the `approach/bilstm` branch of the project repo and runs training there --
the model segfaults on this machine's CPU-only PyTorch build locally (a Windows
OpenMP/MKL threading conflict), and training is much faster on GPU anyway.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/bilstm"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Train

Char-level BiLSTM trained with a masked-language-model objective: randomly
mask letters in each training word, predict the true letter at each masked
position from bidirectional context. At inference this becomes: feed the
real board mask through the model, sum per-position letter probabilities
across all blanks, guess the highest-scoring unguessed letter.

Bump `--epochs` up now that we're on GPU -- 6 was chosen for a slow CPU
sanity check, not a converged run.

In [ ]:
!python src/train_bilstm.py --epochs 20

## Validate

Same methodology as the classical candidate-filtering + n-gram approach
(`approach/candidate-ngram` branch), for a fair comparison: hold out 10% of
train.txt, play full interactive games against words the model never
trained on.

In [ ]:
!python src/validate_bilstm.py

## Save the checkpoint as a notebook output

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes -- copy the trained weights there
explicitly in case the working directory changes.

In [ ]:
import shutil
shutil.copy("src/bilstm_masker.pt", "/kaggle/working/bilstm_masker.pt")
print("saved to /kaggle/working/bilstm_masker.pt -- download it from the Output tab")